# 🔄 Notebook 4: Backtesting & Validation

## Learning Objectives
By the end of this notebook, you will:
- Understand walk-forward validation
- Implement proper time series backtesting
- Evaluate model stability over time
- Compare different strategies
- Understand the limitations of stock prediction

## What You'll Learn
- Why simple train/test split isn't enough
- How to simulate real trading conditions
- TimeSeriesSplit in scikit-learn
- Performance variance across time periods

---

## Why Backtesting?

### Problem with Simple Train/Test Split:
- Tests on **one time period only**
- Might get lucky with market conditions
- Doesn't show if model is **stable** over time

### Walk-Forward Validation:
Simulates real trading where you:
1. Train on all past data
2. Predict next period
3. Observe results
4. **Retrain** with new data
5. Repeat

```
Fold 1: [Training --------][Test--]
Fold 2: [Training ------------][Test--]
Fold 3: [Training ----------------][Test--]
Fold 4: [Training --------------------][Test--]
Fold 5: [Training ------------------------][Test--]
```

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import sys
sys.path.append('..')
import config
from src.data_collection import load_stock_data
from src.preprocessing import preprocess_stock_data
from src.feature_engineering import engineer_all_features, prepare_ml_data
from src.backtesting import walk_forward_validation, compare_strategies

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Ready for backtesting!")

## Step 1: Load Prepared Data

In [ ]:
# Load feature data
ticker = 'AAPL'
feature_path = config.get_data_path(ticker, 'features')

try:
    data = pd.read_csv(feature_path)
    data['Date'] = pd.to_datetime(data['Date'])
    print(f"✅ Loaded feature data: {data.shape}")
except FileNotFoundError:
    print("Running complete pipeline...")
    raw_data = load_stock_data(ticker, 'raw')
    processed = preprocess_stock_data(raw_data)
    data = engineer_all_features(processed)

# Prepare ML data
X, y, feature_names = prepare_ml_data(data)

print(f"\nDataset: {len(X)} samples, {len(feature_names)} features")
print(f"Date range: {data['Date'].min()} to {data['Date'].max()}")

## Step 2: Understand TimeSeriesSplit

Let's visualize how TimeSeriesSplit divides our data.

In [ ]:
# Create time series split
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# Visualize the splits
fig, ax = plt.subplots(figsize=(14, 6))

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    # Plot training set
    ax.barh(fold, len(train_idx), left=train_idx[0], 
            height=0.8, color='blue', alpha=0.6, label='Train' if fold == 0 else '')
    
    # Plot test set
    ax.barh(fold, len(test_idx), left=test_idx[0], 
            height=0.8, color='red', alpha=0.6, label='Test' if fold == 0 else '')
    
    # Add text
    ax.text(train_idx[0] + len(train_idx)/2, fold, 
            f'Train: {len(train_idx)}', 
            ha='center', va='center', fontsize=10)
    ax.text(test_idx[0] + len(test_idx)/2, fold, 
            f'Test: {len(test_idx)}', 
            ha='center', va='center', fontsize=10)

ax.set_yticks(range(n_splits))
ax.set_yticklabels([f'Fold {i+1}' for i in range(n_splits)])
ax.set_xlabel('Sample Index')
ax.set_title('TimeSeriesSplit Visualization', fontsize=16)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("💡 Notice how:")
print("   - Training set grows with each fold")
print("   - Test set moves forward in time")
print("   - No overlap between train and test")
print("   - This simulates real trading!")

## Step 3: Run Walk-Forward Validation

Now let's actually run the backtesting!

In [ ]:
# Run walk-forward validation
print("Starting walk-forward validation...")
print("This will train and test the model 5 times.")
print("It may take 1-2 minutes...\n")

results = walk_forward_validation(X, y, feature_names, n_splits=5)

## Step 4: Analyze Results Across Folds

In [ ]:
# Display results table
print("\n📊 Backtest Results Summary:")
print("="*70)
print(results)

# Calculate statistics
mean_acc = results['accuracy'].mean()
std_acc = results['accuracy'].std()
min_acc = results['accuracy'].min()
max_acc = results['accuracy'].max()

print("\n📈 Performance Statistics:")
print("="*70)
print(f"Mean Accuracy:    {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Best Fold:        {max_acc:.4f} (Fold {results['accuracy'].idxmax() + 1})")
print(f"Worst Fold:       {min_acc:.4f} (Fold {results['accuracy'].idxmin() + 1})")
print(f"Range:            {max_acc - min_acc:.4f}")

# Interpretation
if std_acc < 0.05:
    print("\n✅ Model is STABLE - consistent across time periods")
elif std_acc < 0.10:
    print("\n✓ Model is MODERATELY STABLE")
else:
    print("\n⚠️ Model is UNSTABLE - performance varies significantly")

## Step 5: Visualize Performance Over Time

In [ ]:
# Plot accuracy across folds
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Accuracy over folds
ax1.plot(results['fold'], results['accuracy'], 
         marker='o', linewidth=2, markersize=10, color='blue')
ax1.axhline(y=mean_acc, color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {mean_acc:.4f}')
ax1.fill_between(results['fold'], 
                 mean_acc - std_acc, 
                 mean_acc + std_acc, 
                 alpha=0.2, color='red', label=f'±1 Std Dev')
ax1.set_xlabel('Fold Number', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Model Accuracy Across Time Periods', fontsize=16)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Plot 2: Train and test sizes
ax2.plot(results['fold'], results['train_size'], 
         marker='s', label='Training Size', linewidth=2, markersize=8)
ax2.plot(results['fold'], results['test_size'], 
         marker='^', label='Test Size', linewidth=2, markersize=8)
ax2.set_xlabel('Fold Number', fontsize=12)
ax2.set_ylabel('Number of Samples', fontsize=12)
ax2.set_title('Data Split Sizes', fontsize=16)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print(f"   - Training set grows from {results['train_size'].min()} to {results['train_size'].max()} samples")
print(f"   - Accuracy range: {min_acc:.2%} to {max_acc:.2%}")
if results['accuracy'].is_monotonic_increasing:
    print("   - Performance IMPROVES over time (more data helps!)")
elif results['accuracy'].is_monotonic_decreasing:
    print("   - Performance DEGRADES over time (model aging?)")
else:
    print("   - Performance is VARIABLE across periods")

## Step 6: Compare Baseline vs Hybrid Model

Let's test whether sentiment features actually help!

In [ ]:
# Compare strategies
print("Comparing price-only vs price+sentiment models...")
print("This will take 2-3 minutes...\n")

comparison = compare_strategies(X, y, feature_names, n_splits=5)

## Step 7: Statistical Significance Test

Is the improvement statistically significant?

In [ ]:
from scipy import stats

# Get results
baseline_accs = comparison['baseline_results']['accuracy'].values
hybrid_accs = comparison['hybrid_results']['accuracy'].values

# Paired t-test
t_stat, p_value = stats.ttest_rel(hybrid_accs, baseline_accs)

print("\n📊 Statistical Significance Test")
print("="*70)
print(f"Baseline mean: {baseline_accs.mean():.4f}")
print(f"Hybrid mean:   {hybrid_accs.mean():.4f}")
print(f"Difference:    {hybrid_accs.mean() - baseline_accs.mean():.4f}")
print(f"\nT-statistic:   {t_stat:.4f}")
print(f"P-value:       {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ Difference is STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("   Sentiment features genuinely help!")
else:
    print("\n⚠️ Difference is NOT statistically significant (p >= 0.05)")
    print("   Improvement might be due to chance")

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(baseline_accs))
width = 0.35

ax.bar(x - width/2, baseline_accs, width, label='Baseline (Price Only)', alpha=0.8)
ax.bar(x + width/2, hybrid_accs, width, label='Hybrid (Price + Sentiment)', alpha=0.8)

ax.set_xlabel('Fold Number', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Baseline vs Hybrid Model Comparison', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(len(baseline_accs))])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Step 8: Risk-Adjusted Performance

Consistency matters! Let's look at the Sharpe-like ratio of our predictions.

In [ ]:
# Calculate performance metrics
def calculate_sharpe_ratio(accuracies):
    """Calculate a Sharpe-like ratio for model performance"""
    excess_return = accuracies.mean() - 0.5  # Excess over random
    volatility = accuracies.std()
    if volatility == 0:
        return 0
    return excess_return / volatility

baseline_sharpe = calculate_sharpe_ratio(baseline_accs)
hybrid_sharpe = calculate_sharpe_ratio(hybrid_accs)

print("\n📊 Risk-Adjusted Performance")
print("="*70)
print("Sharpe-like Ratio (higher is better):")
print(f"Baseline: {baseline_sharpe:.4f}")
print(f"Hybrid:   {hybrid_sharpe:.4f}")

if hybrid_sharpe > baseline_sharpe:
    print("\n✅ Hybrid model has better risk-adjusted returns")
else:
    print("\n⚠️ Baseline has better risk-adjusted returns")

# Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [baseline_accs, hybrid_accs]
bp = ax.boxplot(data_to_plot, labels=['Baseline', 'Hybrid'], 
                patch_artist=True, widths=0.6)

# Color the boxes
colors = ['lightblue', 'lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Performance Distribution Across Folds', fontsize=16)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random Baseline')
ax.legend()

plt.tight_layout()
plt.show()

## Step 9: Real-World Considerations

Let's think about what these results mean for actual trading.

In [ ]:
# Simulate simple trading strategy
def simulate_trading(y_true, y_pred, initial_capital=10000):
    """
    Simulate a simple trading strategy:
    - Buy if predict up, sell if predict down
    - Assume we capture the full daily return
    """
    capital = initial_capital
    portfolio_value = [capital]
    
    for actual, predicted in zip(y_true, y_pred):
        # If we predicted correctly and went long
        if predicted == 1:
            if actual == 1:  # Market went up, we profit
                capital *= 1.01  # Assume 1% gain
            else:  # Market went down, we lose
                capital *= 0.99  # Assume 1% loss
        # If we predicted down (or stayed out)
        # Capital stays same (simplified)
        
        portfolio_value.append(capital)
    
    return np.array(portfolio_value)

# Get predictions from last fold for illustration
tscv = TimeSeriesSplit(n_splits=5)
train_idx, test_idx = list(tscv.split(X))[-1]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Train and predict
model = RandomForestClassifier(**config.RF_PARAMS)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Simulate trading
portfolio = simulate_trading(y_test.values, predictions)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(portfolio, linewidth=2)
plt.axhline(y=10000, color='red', linestyle='--', label='Initial Capital')
plt.title('Simulated Portfolio Value (Last Fold)', fontsize=16)
plt.xlabel('Trading Days', fontsize=12)
plt.ylabel('Portfolio Value ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_value = portfolio[-1]
total_return = (final_value - 10000) / 10000 * 100

print(f"\n💰 Trading Simulation (Simplified):")
print(f"Initial Capital: ${10000:,.2f}")
print(f"Final Value:     ${final_value:,.2f}")
print(f"Total Return:    {total_return:+.2f}%")
print("\n⚠️ Note: This is a HIGHLY simplified simulation!")
print("   Real trading includes:")
print("   - Transaction costs")
print("   - Slippage")
print("   - Market impact")
print("   - Risk management")

## 🎯 Exercise: Try Different Stocks

Run the backtesting on different stocks and compare!

In [ ]:
# YOUR CODE HERE
# Try backtesting on MSFT, GOOGL, or TSLA

tickers_to_test = ['AAPL', 'MSFT', 'GOOGL']
results_dict = {}

for ticker in tickers_to_test:
    # Load data
    # Run walk-forward validation
    # Store results
    pass

# Compare: Which stock is most predictable?

## 📝 Key Takeaways

1. **Walk-forward validation** is the gold standard for time series
2. **Performance varies** across different time periods
3. **Consistency** is as important as average accuracy
4. **Statistical significance** matters when comparing models
5. **Real trading** is much more complex than backtests suggest

## Important Realizations

### What We Learned:
- 📊 60-70% accuracy is realistic for stock prediction
- 📈 Model performance is not constant over time
- 💡 Small improvements can be significant
- ⚠️ Backtests are optimistic vs real trading

### Limitations:
1. **Stock markets are fundamentally unpredictable**
2. **Past performance ≠ future results**
3. **Transaction costs** eat into profits
4. **Market conditions change** (regime shifts)
5. **Overfitting** is always a risk

### What Makes a Good Model:
- ✅ Consistently > 55-60% accuracy
- ✅ Stable across time periods
- ✅ Statistically significant improvement
- ✅ Risk-adjusted returns make sense

## Final Thoughts

This project taught you:
- Complete ML pipeline from data to predictions
- Proper time series validation
- Feature engineering for financial data
- Model evaluation and interpretation
- The reality of stock market prediction

**Remember**: This is for **education**, not actual trading!

---

## 🎓 Congratulations!

You've completed the entire machine learning pipeline!

**You now know how to:**
- Collect and prepare financial data
- Engineer predictive features
- Train machine learning models
- Properly validate time series models
- Interpret results realistically

**Next steps:**
- Implement real NLP sentiment analysis
- Try other ML algorithms (XGBoost, LSTM)
- Add more technical indicators
- Build a real-time prediction system
- Learn about portfolio optimization

**Most importantly:** Keep learning and stay curious! 🚀